# E-commerce Checkout A/B Test
## Day 15 — Tableau Data Preparation

### Objective

This notebook creates a Tableau-ready analytical layer without changing the experiment definitions established on Days 8–14. The main export remains at **one row per eligible mature checkout-exposed user**, which makes Tableau aggregations auditable and prevents event-level double counting.

The dashboard will support four views:

1. ordered checkout funnel by experiment group;
2. purchase conversion by device and experiment group;
3. retained revenue per exposed user by experiment group; and
4. payment-failure, checkout-error, and refund/cancellation guardrails.

Retention and repeat purchase are intentionally excluded: the Week 2 experiment observes a first checkout journey and one attributed order, so those metrics are not supported by this dataset.

### Frozen Analysis Definition

- **Population:** eligible users with a first valid checkout exposure and a complete 24-hour observation window
- **Analysis grain:** one row per mature exposed user
- **Primary metric:** attributed purchase conversion within 24 hours
- **Funnel:** `checkout_view → payment_attempt → purchase`
- **Purchase rule:** purchase must occur after a qualifying payment attempt
- **Treatment effect:** treatment minus control

The notebook reuses the Day 11 SQL rather than rebuilding these rules in Tableau. Tableau is the presentation layer, not the source of truth for population eligibility or attribution.

In [1]:
from pathlib import Path
import sqlite3

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)

START_DIR = Path.cwd().resolve()
PROJECT_ROOT = None

for path in [START_DIR, *START_DIR.parents]:
    if (
        path.name == "week2_checkout_experiment"
        and (path / "data" / "raw").is_dir()
    ):
        PROJECT_ROOT = path
        break

    candidate = path / "week2_checkout_experiment"
    if (candidate / "data" / "raw").is_dir():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the week2_checkout_experiment directory."
    )

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
FUNNEL_SQL_PATH = PROJECT_ROOT / "sql" / "02_funnel_analysis.sql"
ORDER_DATA_CUTOFF = "2026-07-22 23:59:59"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT.name)
print("Raw input:", RAW_DATA_DIR.relative_to(PROJECT_ROOT))
print("Processed output:", PROCESSED_DATA_DIR.relative_to(PROJECT_ROOT))
print("Reused SQL:", FUNNEL_SQL_PATH.relative_to(PROJECT_ROOT))

Project root: week2_checkout_experiment
Raw input: data/raw
Processed output: data/processed
Reused SQL: sql/02_funnel_analysis.sql


## 1. Rebuild the Frozen Experiment Population

The four raw tables have different grains. They are loaded into an in-memory SQLite database, and the Day 11 SQL reconstructs the canonical assignment, valid exposure, mature population, ordered funnel, and attributed purchase.

In [2]:
conn = sqlite3.connect(":memory:")

raw_table_audit = []
for table_name in ["users", "experiment_assignments", "events", "orders"]:
    table = pd.read_csv(RAW_DATA_DIR / f"{table_name}.csv")
    table.to_sql(table_name, conn, index=False, if_exists="replace")
    raw_table_audit.append({"table_name": table_name, "rows": len(table)})

conn.executescript(FUNNEL_SQL_PATH.read_text(encoding="utf-8"))

print(pd.DataFrame(raw_table_audit).to_string(index=False))
print("\nDay 11 population and funnel rebuilt successfully.")

            table_name  rows
                 users 20000
experiment_assignments 20060
                events 36813
                orders  5043

Day 11 population and funnel rebuilt successfully.


## 2. Add Day 14 Supporting and Guardrail Metrics

The supporting fields reuse the same denominators as Day 14:

- payment failure is evaluated among payment-attempt users;
- checkout error is evaluated among mature exposed users;
- refund/cancellation is evaluated among attributed purchasers with mature follow-up; and
- retained revenue assigns completed-order revenue to purchasers and zero to all other exposed users.

In [3]:
supporting_sql = f"""
CREATE TEMP TABLE exposure_sessions AS
SELECT
    f.user_id,
    MIN(e.session_id) AS exposure_session_id
FROM user_level_funnel f
JOIN deduplicated_events e
    ON f.user_id = e.user_id
   AND f.experiment_group = e.experiment_group
   AND e.event_name = 'checkout_view'
   AND e.event_timestamp = f.exposure_timestamp
GROUP BY f.user_id;

CREATE TEMP TABLE user_guardrail_flags AS
SELECT
    f.user_id,
    MAX(
        CASE
            WHEN e.event_name = 'payment_failure'
             AND f.payment_attempt_timestamp IS NOT NULL
             AND datetime(e.event_timestamp) >= datetime(f.payment_attempt_timestamp)
             AND datetime(e.event_timestamp) <= datetime(f.exposure_timestamp, '+24 hours')
            THEN 1 ELSE 0
        END
    ) AS had_payment_failure,
    MAX(
        CASE
            WHEN e.event_name = 'checkout_error'
             AND e.session_id = s.exposure_session_id
             AND datetime(e.event_timestamp) >= datetime(f.exposure_timestamp)
            THEN 1 ELSE 0
        END
    ) AS had_checkout_error
FROM user_level_funnel f
JOIN exposure_sessions s
    ON f.user_id = s.user_id
LEFT JOIN deduplicated_events e
    ON f.user_id = e.user_id
   AND f.experiment_group = e.experiment_group
GROUP BY f.user_id;
"""

conn.executescript(supporting_sql)
print("Day 14 supporting and guardrail definitions attached.")

Day 14 supporting and guardrail definitions attached.


In [4]:
tableau_user_metrics = pd.read_sql_query(
    f"""
    SELECT
        f.user_id,
        f.experiment_group,
        f.assignment_timestamp,
        f.exposure_timestamp,
        f.device_at_exposure,
        f.traffic_source_at_exposure,
        f.user_type,
        f.payment_attempt_timestamp,
        f.purchase_timestamp,
        f.reached_checkout_view,
        f.reached_payment_attempt,
        f.reached_purchase,
        g.had_payment_failure,
        g.had_checkout_error,
        CASE
            WHEN f.reached_payment_attempt = 1 THEN 1 - g.had_payment_failure
        END AS payment_succeeded,
        CASE
            WHEN f.purchase_timestamp IS NOT NULL
            THEN (julianday(f.purchase_timestamp) - julianday(f.exposure_timestamp)) * 24 * 60
        END AS completion_minutes,
        o.order_id,
        o.order_amount,
        o.order_status,
        CASE
            WHEN f.purchase_timestamp IS NOT NULL
             AND datetime(f.purchase_timestamp, '+7 days') <= datetime('{ORDER_DATA_CUTOFF}')
            THEN 1 ELSE 0
        END AS mature_order_followup,
        CASE WHEN o.order_status = 'completed' THEN o.order_amount ELSE 0 END AS retained_revenue,
        CASE WHEN o.order_status IN ('refunded', 'cancelled') THEN 1 ELSE 0 END AS refunded_or_cancelled
    FROM user_level_funnel f
    LEFT JOIN user_guardrail_flags g
        ON f.user_id = g.user_id
    LEFT JOIN orders o
        ON f.user_id = o.user_id
       AND f.experiment_group = o.experiment_group
       AND f.purchase_timestamp = o.purchase_timestamp
    ORDER BY f.user_id;
    """,
    conn,
)

tableau_user_metrics.insert(
    4,
    "exposure_date",
    pd.to_datetime(tableau_user_metrics["exposure_timestamp"]).dt.strftime("%Y-%m-%d"),
)

tableau_user_metrics.head()

,user_id,experiment_group,assignment_timestamp,exposure_timestamp,exposure_date,device_at_exposure,traffic_source_at_exposure,user_type,payment_attempt_timestamp,purchase_timestamp,reached_checkout_view,reached_payment_attempt,reached_purchase,had_payment_failure,had_checkout_error,payment_succeeded,completion_minutes,order_id,order_amount,order_status,mature_order_followup,retained_revenue,refunded_or_cancelled
0,U000001,control,2026-07-10 01:48:57,2026-07-10 03:55:57.474342798,2026-07-10,mobile,direct,returning,2026-07-10 04:17:57.474342798,None,1,1,0,0,0,1.0,NaN,None,NaN,None,0,0.0,0
1,U000002,treatment,2026-07-08 17:33:58,2026-07-08 19:53:25.247435122,2026-07-08,desktop,direct,returning,None,None,1,0,0,0,0,NaN,NaN,None,NaN,None,0,0.0,0
2,U000003,control,2026-07-12 10:15:32,2026-07-12 14:02:33.743649830,2026-07-12,mobile,direct,new,None,None,1,0,0,0,0,NaN,NaN,None,NaN,None,0,0.0,0
3,U000004,treatment,2026-07-09 12:50:32,2026-07-09 15:21:40.318484834,2026-07-09,desktop,paid_search,returning,2026-07-09 15:25:40.318484834,None,1,1,0,0,0,1.0,NaN,None,NaN,None,0,0.0,0
4,U000005,control,2026-07-14 05:39:05,2026-07-14 08:46:27.258509648,2026-07-14,desktop,paid_search,new,2026-07-14 08:59:27.258509648,None,1,1,0,0,0,1.0,NaN,None,NaN,None,0,0.0,0


## 3. Validate the Tableau User Grain

Tableau will aggregate this file. A duplicate user would inflate every count and rate, so the grain is validated before export. The expected figures are frozen to the Day 11–14 results.

In [5]:
expected_group_sizes = {"control": 7_754, "treatment": 7_713}
expected_purchases = {"control": 2_130, "treatment": 2_285}

assert len(tableau_user_metrics) == 15_467
assert tableau_user_metrics["user_id"].nunique() == len(tableau_user_metrics)
assert not tableau_user_metrics["user_id"].duplicated().any()
assert set(tableau_user_metrics["experiment_group"]) == {"control", "treatment"}
assert set(tableau_user_metrics["reached_purchase"]) <= {0, 1}
assert not tableau_user_metrics[["user_type", "device_at_exposure", "traffic_source_at_exposure"]].isna().any().any()
assert tableau_user_metrics.groupby("experiment_group").size().to_dict() == expected_group_sizes
assert tableau_user_metrics.groupby("experiment_group")["reached_purchase"].sum().to_dict() == expected_purchases
assert tableau_user_metrics["reached_purchase"].sum() == 4_415
assert tableau_user_metrics["order_id"].notna().sum() == 4_415
assert tableau_user_metrics.loc[tableau_user_metrics["reached_purchase"].eq(1), "mature_order_followup"].eq(1).all()

grain_audit = pd.DataFrame(
    {
        "check": [
            "analysis rows",
            "unique users",
            "duplicate users",
            "attributed purchases",
            "matched orders",
        ],
        "value": [
            len(tableau_user_metrics),
            tableau_user_metrics["user_id"].nunique(),
            tableau_user_metrics["user_id"].duplicated().sum(),
            tableau_user_metrics["reached_purchase"].sum(),
            tableau_user_metrics["order_id"].notna().sum(),
        ],
    }
)
print(grain_audit.to_string(index=False))

               check  value
       analysis rows  15467
        unique users  15467
     duplicate users      0
attributed purchases   4415
      matched orders   4415


## 4. Create a Separate Funnel Summary

The funnel export is kept separate because it has a different grain: one row per experiment group and funnel step. Joining it to the user-level file would duplicate summary values. In Tableau, it will be added as a second data source for the funnel worksheet only.

In [6]:
tableau_funnel_summary = pd.read_sql_query(
    """
    SELECT
        experiment_group,
        step_order,
        funnel_step,
        step_users,
        previous_step_users,
        step_to_step_conversion,
        conversion_from_exposure
    FROM funnel_by_step
    ORDER BY experiment_group, step_order;
    """,
    conn,
)

step_labels = {
    1: "1. Checkout View",
    2: "2. Payment Attempt",
    3: "3. Purchase",
}
tableau_funnel_summary.insert(
    3,
    "funnel_step_label",
    tableau_funnel_summary["step_order"].map(step_labels),
)
tableau_funnel_summary["drop_off_from_previous"] = (
    tableau_funnel_summary["previous_step_users"]
    - tableau_funnel_summary["step_users"]
)

tableau_funnel_summary

,experiment_group,step_order,funnel_step,funnel_step_label,step_users,previous_step_users,step_to_step_conversion,conversion_from_exposure,drop_off_from_previous
0,control,1,checkout_view,1. Checkout View,7754,7754,1.0000,1.0000,0
1,control,2,payment_attempt,2. Payment Attempt,5844,7754,0.7537,0.7537,1910
2,control,3,purchase,3. Purchase,2130,5844,0.3645,0.2747,3714
3,treatment,1,checkout_view,1. Checkout View,7713,7713,1.0000,1.0000,0
4,treatment,2,payment_attempt,2. Payment Attempt,6004,7713,0.7784,0.7784,1709
5,treatment,3,purchase,3. Purchase,2285,6004,0.3806,0.2963,3719


## 5. Reconcile the Dashboard KPIs

This table is a validation checkpoint. Tableau should reproduce these values from the exported user-level file.

In [7]:
kpi_rows = []
for experiment_group, group_data in tableau_user_metrics.groupby("experiment_group"):
    attempted = group_data.loc[group_data["reached_payment_attempt"].eq(1)]
    purchasers = group_data.loc[group_data["reached_purchase"].eq(1)]

    kpi_rows.append(
        {
            "experiment_group": experiment_group,
            "exposed_users": len(group_data),
            "payment_attempt_users": int(group_data["reached_payment_attempt"].sum()),
            "purchase_users": int(group_data["reached_purchase"].sum()),
            "purchase_conversion": group_data["reached_purchase"].mean(),
            "payment_failure_rate": attempted["had_payment_failure"].mean(),
            "checkout_error_rate": group_data["had_checkout_error"].mean(),
            "refund_cancel_rate": purchasers["refunded_or_cancelled"].mean(),
            "retained_revenue_per_exposed": group_data["retained_revenue"].mean(),
            "median_completion_minutes": purchasers["completion_minutes"].median(),
        }
    )

tableau_kpi_summary = pd.DataFrame(kpi_rows).sort_values("experiment_group")

assert tableau_kpi_summary.set_index("experiment_group")["purchase_conversion"].round(4).to_dict() == {
    "control": 0.2747,
    "treatment": 0.2963,
}
assert tableau_kpi_summary.set_index("experiment_group")["retained_revenue_per_exposed"].round(2).to_dict() == {
    "control": 24.52,
    "treatment": 26.33,
}

kpi_display = tableau_kpi_summary.copy()
for column in ["purchase_conversion", "payment_failure_rate", "checkout_error_rate", "refund_cancel_rate"]:
    kpi_display[column] = kpi_display[column].map("{:.2%}".format)
kpi_display["retained_revenue_per_exposed"] = kpi_display["retained_revenue_per_exposed"].map(lambda value: f"${value:.2f}")
kpi_display["median_completion_minutes"] = kpi_display["median_completion_minutes"].map("{:.2f}".format)

print(kpi_display.to_string(index=False))

experiment_group  exposed_users  payment_attempt_users  purchase_users purchase_conversion payment_failure_rate checkout_error_rate refund_cancel_rate retained_revenue_per_exposed median_completion_minutes
         control           7754                   5844            2130              27.47%                5.36%               3.07%              5.68%                       $24.52                     28.13
       treatment           7713                   6004            2285              29.63%                5.13%               3.49%              6.48%                       $26.33                     26.82


## 6. Export Tableau-Ready Files

Three CSV files are produced:

- `tableau_user_metrics.csv`: the primary Tableau source, one row per mature exposed user;
- `tableau_funnel_summary.csv`: a six-row funnel source; and
- `tableau_data_dictionary.csv`: field definitions and grains.

In [8]:
data_dictionary_rows = [
    ("tableau_user_metrics", "user_id", "Unique mature exposed user identifier", "one row per mature exposed user"),
    ("tableau_user_metrics", "experiment_group", "Persistent assigned variant: control or treatment", "one row per mature exposed user"),
    ("tableau_user_metrics", "assignment_timestamp", "Canonical experiment assignment time", "one row per mature exposed user"),
    ("tableau_user_metrics", "exposure_timestamp", "First valid checkout view after assignment", "one row per mature exposed user"),
    ("tableau_user_metrics", "exposure_date", "Calendar date of first valid exposure", "one row per mature exposed user"),
    ("tableau_user_metrics", "device_at_exposure", "Device recorded at first valid exposure", "one row per mature exposed user"),
    ("tableau_user_metrics", "traffic_source_at_exposure", "Traffic source recorded at first valid exposure", "one row per mature exposed user"),
    ("tableau_user_metrics", "user_type", "New or returning user status", "one row per mature exposed user"),
    ("tableau_user_metrics", "payment_attempt_timestamp", "First qualifying payment attempt within 24 hours", "one row per mature exposed user"),
    ("tableau_user_metrics", "purchase_timestamp", "First attributed purchase after payment attempt and within 24 hours", "one row per mature exposed user"),
    ("tableau_user_metrics", "reached_checkout_view", "1 for every included exposed user", "one row per mature exposed user"),
    ("tableau_user_metrics", "reached_payment_attempt", "1 if user reached a qualifying payment attempt", "one row per mature exposed user"),
    ("tableau_user_metrics", "reached_purchase", "1 if user completed an attributed purchase", "one row per mature exposed user"),
    ("tableau_user_metrics", "had_payment_failure", "1 if an attempted user recorded a payment failure in the 24-hour window", "one row per mature exposed user"),
    ("tableau_user_metrics", "had_checkout_error", "1 if user recorded an error in the first valid exposure session", "one row per mature exposed user"),
    ("tableau_user_metrics", "payment_succeeded", "1 minus payment failure for payment-attempt users; null otherwise", "one row per mature exposed user"),
    ("tableau_user_metrics", "completion_minutes", "Minutes from exposure to attributed purchase; null for non-purchasers", "one row per mature exposed user"),
    ("tableau_user_metrics", "order_id", "Attributed order identifier; null for non-purchasers", "one row per mature exposed user"),
    ("tableau_user_metrics", "order_amount", "Gross attributed order value; null for non-purchasers", "one row per mature exposed user"),
    ("tableau_user_metrics", "order_status", "Final order status after seven-day follow-up", "one row per mature exposed user"),
    ("tableau_user_metrics", "mature_order_followup", "1 when a purchaser has complete seven-day status follow-up", "one row per mature exposed user"),
    ("tableau_user_metrics", "retained_revenue", "Completed-order revenue, otherwise zero", "one row per mature exposed user"),
    ("tableau_user_metrics", "refunded_or_cancelled", "1 for an attributed refunded or cancelled order", "one row per mature exposed user"),
    ("tableau_funnel_summary", "experiment_group", "Control or treatment variant", "one row per experiment group and funnel step"),
    ("tableau_funnel_summary", "step_order", "Numeric funnel sequence", "one row per experiment group and funnel step"),
    ("tableau_funnel_summary", "funnel_step", "Machine-readable funnel step", "one row per experiment group and funnel step"),
    ("tableau_funnel_summary", "funnel_step_label", "Display label with stable sort prefix", "one row per experiment group and funnel step"),
    ("tableau_funnel_summary", "step_users", "Users reaching the cumulative funnel step", "one row per experiment group and funnel step"),
    ("tableau_funnel_summary", "previous_step_users", "Users reaching the immediately prior step", "one row per experiment group and funnel step"),
    ("tableau_funnel_summary", "step_to_step_conversion", "Step users divided by previous-step users", "one row per experiment group and funnel step"),
    ("tableau_funnel_summary", "conversion_from_exposure", "Step users divided by exposed users", "one row per experiment group and funnel step"),
    ("tableau_funnel_summary", "drop_off_from_previous", "Previous-step users minus step users", "one row per experiment group and funnel step"),
]

tableau_data_dictionary = pd.DataFrame(
    data_dictionary_rows,
    columns=["table_name", "column_name", "description", "grain_note"],
)

output_paths = {
    "user metrics": PROCESSED_DATA_DIR / "tableau_user_metrics.csv",
    "funnel summary": PROCESSED_DATA_DIR / "tableau_funnel_summary.csv",
    "data dictionary": PROCESSED_DATA_DIR / "tableau_data_dictionary.csv",
}

tableau_user_metrics.to_csv(output_paths["user metrics"], index=False)
tableau_funnel_summary.to_csv(output_paths["funnel summary"], index=False)
tableau_data_dictionary.to_csv(output_paths["data dictionary"], index=False)

export_audit = pd.DataFrame(
    [
        {"output": label, "rows": len(pd.read_csv(path)), "path": str(path.relative_to(PROJECT_ROOT))}
        for label, path in output_paths.items()
    ]
)
print(export_audit.to_string(index=False))

conn.close()

         output  rows                                       path
   user metrics 15467    data/processed/tableau_user_metrics.csv
 funnel summary     6  data/processed/tableau_funnel_summary.csv
data dictionary    32 data/processed/tableau_data_dictionary.csv


## Tableau Calculation Map

Create these calculated fields from `tableau_user_metrics.csv`:

```text
Purchase Conversion
AVG([reached_purchase])

Retained Revenue per Exposed User
AVG([retained_revenue])

Payment Failure Rate
SUM([had_payment_failure]) / SUM([reached_payment_attempt])

Checkout Error Rate
AVG([had_checkout_error])

Refund / Cancellation Rate
SUM([refunded_or_cancelled]) / SUM([reached_purchase])
```

The denominators are intentionally different. Never replace all three guardrails with a simple average unless the field definition supports the full exposed-user denominator.

## Expected Dashboard Checkpoints

Your Tableau workbook should reproduce the following values:

- mature exposed users: **15,467**;
- control users: **7,754**; treatment users: **7,713**;
- control purchase conversion: **27.47%**;
- treatment purchase conversion: **29.63%**;
- retained revenue per exposed user: **$24.52 control**, **$26.33 treatment**;
- payment failure: **5.36% control**, **5.13% treatment**;
- checkout error: **3.07% control**, **3.49% treatment**; and
- refund/cancellation: **5.68% control**, **6.48% treatment**.

If Tableau does not match these figures, stop and inspect the aggregation and denominator before formatting the dashboard.